# Notebook19 — source-bound three-arm JEPA runner

Purpose: run the original ViT, mathematically equivalent optimized learned kernel (local gate0.5), and the same kernel with local gate0.1. Inputs are frozen TRAIN/DEV memberships; official pixels remain closed. Production uses two fresh seeds, each complete triplet stays on one registered GPU,256px FP32, physical16× accumulation8,4ordered workers,4000-update scheduler horizon, and900active seconds per arm.

The same training pass pins the exact200-update full-state milestone outside rolling latest-two/best retention. Initial, matched200, and final DEV linear/kNN probes are descriptive. If200updates are not reached, preserve UNKNOWN; never substitute another step. Runtime is bounded by the parent study's original three-hour deadline. Checkpoints are under`checkpoints/<run_id>/stages/<stage_id>/arms/<arm_id>/`, with immutable`milestones/step_000200.pt`.

Definitions load only through`ahr-export`. Exact resume uses the main notebook's launch cell; READY timers continue from verified full state, while IN_FLIGHT timers remain UNKNOWN rather than resetting elapsed work. RunAll here imports definitions only and never launches training.

In [ ]:
"""Source-bound learned positive kernel runner; equal active-wall full JEPA and exact resume."""
from pathlib import Path
import copy,gc,json,math,os,time
import nbformat,torch
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/19_compute_and_gate/runner.ipynb'))
_PIPELINE=_SELF.resolve().parents[1]/'17_fixed_kernel/pipeline.ipynb'
_ns={'__file__':str(_PIPELINE)}
for _cell in nbformat.read(_PIPELINE,4).cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_PIPELINE),'exec'),_ns)
C=type('Pipeline',(),_ns)
_ORIGINAL_SOURCE_HASHES=C.source_hashes

def source_hashes(root):
    root=Path(root).resolve();identity=_ORIGINAL_SOURCE_HASHES(root)
    for path in sorted([*(root/'experiments/18_learned_kernel').glob('*.ipynb'),*(root/'experiments/19_compute_and_gate').glob('*.ipynb')]):
        identity[str(path.relative_to(root))]=C.export_hash(path)
    return identity

C.source_hashes=source_hashes

RESUME_NOTEBOOK='notebooks/19_Compute_Optimization_and_Global_Prior_Pilot.ipynb'
RESUME_CELL='launch-compute-gate-pilot'
VARIANTS=('vit','learned_optimized','learned_global_prior')

def modules(root):
    """Isolated tagged15/05 instances: source discovery cannot rebind the active18 worker."""
    import sys,types
    root=Path(root).resolve();C.bootstrap(root);name='fmm19_isolated_runner15'
    if name not in sys.modules:
        path=root/'experiments/15_vit_geometry/runner.ipynb';runner=types.ModuleType(name);runner.__file__=str(path);sys.modules[name]=runner
        try:
            for cell in nbformat.read(path,4).cells:
                if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
                    definition=cell.source.replace("'fmm15_isolated_inherited_runner05'","'fmm19_isolated_inherited_runner05'")
                    exec(compile(definition,str(path)+'#isolated-ahr-export','exec'),runner.__dict__)
        except BaseException:sys.modules.pop(name,None);raise
    runner=sys.modules[name]
    if Path(runner.__file__).resolve()!=root/'experiments/15_vit_geometry/runner.ipynb':raise ValueError('Isolated15 path changed')
    return runner,C.load(root,'experiments/15_vit_geometry/encoder.ipynb','fmm15_encoder'),C.load(root,'experiments/19_compute_and_gate/operator.ipynb','fmm19_operator')

def selected_for_variant(selected,variant):
    if variant not in VARIANTS:raise ValueError('One of the three registered19 arms required')
    value=C.canonical(selected);value['gate_initial']=.1 if variant=='learned_global_prior' else .5
    return value

def build_model(root,config,kind,selected):
    runner,encoder,operator=modules(root)
    variant=config.get('experiment_variant19')
    if variant not in VARIANTS or kind!=('vit' if variant=='vit' else 'semantic'):raise ValueError('Frozen19 architecture dispatch differs')
    expected=selected_for_variant(selected,variant)
    if config.get('learned_kernel19')!=expected:raise ValueError('Selected19 options or predeclared gate differ from configuration')
    expected_base='vit' if variant=='vit' else 'optional_geometry'
    if config.get('kernel15',{}).get('variant')!=expected_base:raise ValueError('Frozen15 scaffold dispatch differs')
    model=runner._default_build_model(config,kind)
    if kind=='semantic':
        options={k:v for k,v in expected.items()if k in('rank','feature_seed','local_radius','spatial_mode','feature_temperature_min','feature_temperature_max','feature_temperature_initial','gate_initial')}
        model=operator.replace_encoder_mixers(model,options)
    return model

def builder(root,selected):return lambda config,kind:build_model(root,config,kind,selected)

def _persist_numerical_backend_stats(artifact,stage_id,arm_id,event):
    value={'status':'RECORDED','backend':'Math-preserving optimized learned positive global features plus exact local softmax convex mixture',
        'stage_id':stage_id,'arm_id':arm_id,'event':event,'updated_unix':time.time(),'worker_pid':os.getpid(),
        'operator':'experiments/19_compute_and_gate/operator.ipynb','theoretical_scope':'Exact summaries for the learned finite kernel; no softmax approximation guarantee'}
    C.atomic_json(Path(artifact)/'numerical_backend_stats.json',value);return value

def bind(root,selected):
    runner,encoder,operator=modules(root);identity=C.source_hashes(root)
    runner.configure_input_pipeline=lambda config:C.install_input_pipeline(int(config.get('input_pipeline',{}).get('workers',0)),root)
    runner.bind_model_builder(builder(root,selected),identity)
    runner._plan_binding=lambda root_value,plan,bundle:plan_binding(root_value,plan,bundle,runner)
    runner._BASE._plan_binding=runner._plan_binding
    runner._persist_numerical_backend_stats=_persist_numerical_backend_stats
    runner._BASE._persist_numerical_backend_stats=_persist_numerical_backend_stats
    runner.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner.RESUME_CELL=RESUME_CELL
    runner._BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner._BASE.RESUME_CELL=RESUME_CELL
    return runner,identity

def data_bundle(root,artifact,config,verify_images=True):
    root,artifact=Path(root).resolve(),Path(artifact).resolve();C.bootstrap(root)
    module=C.load(root,'experiments/05_scaling/preparation.ipynb','fmm19_data05')
    def paths(source,folder,stage):
        folder=Path(folder).resolve();relative=folder.relative_to(root/'artifacts/notebook_19');run_id=relative.parts[0]
        if not run_id or any(not(c.isalnum() or c in '-_')for c in run_id):raise ValueError('Safe19 run ID required')
        return folder,root/'checkpoints'/run_id/'data',run_id
    module._stage_paths=paths
    resolution=int(config['image_size'])
    if resolution not in (128,256,512):raise ValueError('Registered19 data stages are128,256,512')
    if resolution==512:module.STAGE_DEFINITIONS['fixed512']={'id':'fixed512','image_size':512,'pretrain_pool':'selected'}
    stage={'id':('fixed512' if resolution==512 else 'resolution' if resolution==256 else 'learning'),'image_size':resolution,'pretrain_pool':'selected'}
    return module.prepare_stage_bundle(root,artifact,stage,config,verify_images=verify_images)

def _base_config(root):
    _,config,source=C.learned_source(root,'vit');return config

def make_plan(root,run_id,proposal,selected,devices=('cuda:0','cuda:1'),stage_id='pilot',input_workers=4,deadline=None,gate_receipts=None):
    """Three paired arms: identical common state, gate=.5 versus=.1 only mathematical change."""
    root=Path(root).resolve();base=_base_config(root);evaluation=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm19_plan_evaluation')
    probe=evaluation.probe_configuration(base.get('probe_convex'));seeds=list(map(int,proposal['seeds']));updates=int(proposal.get('schedule_updates',4000));arms=[]
    matched=int(proposal.get('matched_updates',200))
    if not seeds or len(seeds)!=len(set(seeds)):raise ValueError('Unique fresh paired seeds required')
    if not 0<matched<updates:raise ValueError('Matched endpoint must precede unchanged scheduler horizon')
    if int(input_workers)not in(0,4,8):raise ValueError('Frozen pipeline workers0,4,8 required')
    if not devices or len(devices)!=len(set(devices))or len(devices)>len(seeds):raise ValueError('Unique explicit devices and whole seed triplets required')
    for number,seed in enumerate(seeds):
        order=VARIANTS if number%2==0 else tuple(reversed(VARIANTS))
        for variant in order:
            cfg=copy.deepcopy(base);cfg.update(image_size=int(proposal.get('image_size',256)),device=str(devices[number%len(devices)]),dtype='float32',cpu_threads=2)
            cfg['seeds'].update(model_init=seed,data_order=seed+101,augmentation=seed+201,mask=seed+301)
            cfg['training'].update(updates=updates,microbatch=16,accumulation_steps=8,checkpoint_every=50)
            hard=float(proposal.get('hard_limit_seconds',10800));cfg['budget'].update(max_seconds=hard,target_seconds=hard,round_updates=50)
            cfg['input_pipeline']={'workers':int(input_workers),'ordered':True,'prefetch':False};cfg['execution_optimization']=False
            cfg['experiment_variant19']=variant;cfg['learned_kernel19']=selected_for_variant(selected,variant)
            cfg['evaluation']['heldout_enabled']=False;cfg['evaluation']['batch_size']=32;cfg['probe_convex']=copy.deepcopy(probe)
            cfg['kernel15'].update(variant='vit'if variant=='vit'else'optional_geometry',temperature_trainable=False,spatial_length_trainable=False,spatial_coefficient_trainable=False)
            cfg.pop('kernel14',None)
            arms.append({'id':'s'+str(seed)+'-'+variant,'seed':seed,'variant':variant,'condition':variant,'kind':'vit'if variant=='vit'else'semantic','config':cfg})
    main=root/RESUME_NOTEBOOK;code=[''.join(c['source'])for c in json.loads(main.read_text())['cells']if c['cell_type']=='code']if main.exists()else None
    return {'schema':'notebook19-three-arm-equal-active-wall-plan/v1','run_id':run_id,'stage_id':stage_id,'seeds':seeds,'registered_seeds':seeds,'variants':list(VARIANTS),'arms':arms,
        'updates':updates,'milestones':[matched,updates],'matched_updates':matched,'round_updates':50,'effective_batch':128,
        'probe_endpoints':['initial','matched','final'],'probe_convex':probe,'train_count':10100,'probe_train_count':10100,
        'hard_limit_seconds':float(proposal.get('hard_limit_seconds',10800)),
        'active_wall_seconds_per_arm':float(proposal['active_wall_seconds_per_arm']),'wall_tolerance_fraction':.02,'wall_tolerance_seconds':2.,
        'active_wall_scope':'Invocation includes input preparation, model load, synchronization, checkpoint/metrics and pinned matched-update writes, and cleanup. Atomic elapsed-time publication excluded; downtime excluded only after committed READY journal',
        'crash_policy':'IN_FLIGHT on restart remains UNKNOWN fairness; full-state checkpoints remain resumable but missing wall time is never reset',
        'selected':C.canonical(selected),'input_workers':int(input_workers),'source_hashes':C.source_hashes(root),'gate_receipts':gate_receipts or [],'deadline_unix':deadline,
        'resume_notebook':RESUME_NOTEBOOK,'resume_cell':RESUME_CELL,'consumer_notebook_sha256':C.digest(code)if code is not None else None,
        'official_test':'CLOSED','devices':list(map(str,devices)),'gpu_workers':len(devices),
        'candidate_display':'Same optimized learned positive kernel; local sigmoid gate initialized .5 or .1',
        'training_gate':'Independent numerical validity and exact full-state resume; no teacher-fidelity, speed, or DEV quality gate',
        'arm_order':'One complete triplet per seed/device; second seed reverses first architecture order',
        'development_scope':'Exploratory two-seed pilot; no statistical significance or official confirmation claim',
        'matched_endpoint_policy':'Pin exact200-update full-state checkpoint during the same900-second training pass; preserve UNKNOWN if not reached'}

def make_screen_plan(root,run_id,selected,devices=('cuda:0','cuda:1'),deadline=None,input_workers=4,gate_receipts=None):
    if int(selected.get('image_size',256))!=256:raise ValueError('Notebook19 registers256px learning only')
    proposal={'seeds':[20262201,20262202],'schedule_updates':4000,'matched_updates':200,
        'active_wall_seconds_per_arm':900.,'hard_limit_seconds':10800.,'image_size':256}
    return make_plan(root,run_id,proposal,selected,devices,'pilot',input_workers,deadline,gate_receipts)


def plan_binding(root,plan,bundle,backend):
    """Local19 replacement for frozen15's two-architecture validation; no upstream mutation."""
    root=Path(root).resolve();safe=lambda value:isinstance(value,str)and bool(value)and all(c.isalnum()or c in'-_'for c in value)
    if not safe(plan.get('run_id'))or not safe(plan.get('stage_id')):raise ValueError('Safe run and stage identifiers required')
    arms=plan.get('arms',[])
    if not arms or len({a['id']for a in arms})!=len(arms)or any(not safe(a['id'])for a in arms):raise ValueError('Unique safe arm IDs required')
    updates=int(plan['updates']);matched=int(plan['matched_updates']);milestones=list(map(int,plan['milestones']))
    if updates<4 or not 0<matched<updates or milestones!=[matched,updates]:raise ValueError('One fixed matched endpoint and unchanged full scheduler horizon required')
    if int(plan['round_updates'])<1:raise ValueError('Positive round size required')
    if tuple(plan['variants'])!=VARIANTS or plan['probe_endpoints']!=['initial','matched','final']:raise ValueError('Exactly three registered arms/endpoints required')
    if not 0<float(plan['hard_limit_seconds'])<=10800:raise ValueError('Bounded at three hours required')
    train=list(bundle['train_ids']);probe=list(bundle['probe_train_ids']);dev=list(bundle['dev_ids']);heldout=list(bundle.get('heldout_ids',[]))
    if not train or not probe or not dev or any(len(ids)!=len(set(ids))for ids in(train,probe,dev)):raise ValueError('Unique nonempty memberships required')
    if set(train)&set(dev)or set(probe)&set(dev)or(set(train)|set(probe)|set(dev))&set(heldout):raise ValueError('Development/official-test leakage')
    if not set(probe).issubset(train):raise ValueError('TRAIN-only probe fitting required')
    for name,ids in[('train_count',train),('probe_train_count',probe)]:
        if int(plan[name])!=len(ids):raise ValueError('Registered membership differs:'+name)
    seeds=sorted({int(a['seed'])for a in arms})
    if sorted(map(int,plan['seeds']))!=seeds or not set(seeds).issubset(set(map(int,plan['registered_seeds']))):raise ValueError('Seed membership differs')
    for seed in seeds:
        group=[a for a in arms if int(a['seed'])==seed]
        if sorted(a['variant']for a in group)!=sorted(VARIANTS):raise ValueError('Each seed requires all three arms')
        common=[]
        for arm in group:
            config=backend.normalize_stage_plan(arm['config']);variant=arm['variant'];policy=backend._strict_training_protocol(config)
            if arm['kind']!=('vit'if variant=='vit'else'semantic')or int(config['seeds']['model_init'])!=seed or config.get('experiment_variant19')!=variant:raise ValueError('Architecture/variant/seed differs')
            if config.get('learned_kernel19')!=selected_for_variant(plan['selected'],variant):raise ValueError('Options/gate differ from registered mathematical variant')
            if policy['updates']!=updates or policy['effective_batch']!=int(plan['effective_batch']):raise ValueError('Training horizon or effective batch differs')
            if config.get('dtype')!='float32'or config.get('evaluation',{}).get('heldout_enabled',False)or int(config.get('cpu_threads',2))!=2:raise ValueError('FP32 deterministic closed-test configuration required')
            if policy['checkpoint_every']>50:raise ValueError('Checkpoint at most every50 updates required')
            expected='vit'if variant=='vit'else'optional_geometry'
            if config.get('kernel15',{}).get('variant')!=expected:raise ValueError('Frozen15 scaffold dispatch differs')
            config['kernel15'].pop('variant');config.pop('experiment_variant19');config.pop('learned_kernel19');config.get('model',{}).pop('kind',None);common.append(config)
        if any(c!=common[0]for c in common[1:]):raise ValueError('Triplet configuration differs beyond architecture and predeclared gate')
    sources=plan['source_hashes']
    if sources!=source_hashes(root):raise ValueError('Definition source hashes changed')
    if plan.get('consumer_notebook_sha256'):
        notebook=json.loads((root/plan['resume_notebook']).read_text());code=[''.join(c['source'])for c in notebook['cells']if c['cell_type']=='code']
        if C.digest(code)!=plan['consumer_notebook_sha256']:raise ValueError('Registered frontdoor code changed')
    return {'schema':'notebook19-stage-binding/v1','plan_hash':C.digest(plan),'source_hashes':sources,
        'dataset_fingerprint':bundle['dataset'].fingerprint,'split_sha256':bundle['split_manifest']['split_sha256'],
        'training_ids_sha256':C.digest(train),'training_examples':len(train),'probe_training_ids_sha256':C.digest(probe),'probe_training_examples':len(probe),
        'development_ids_sha256':C.digest(dev),'development_examples':len(dev),'heldout_status':'NOT_RUN'}


def worker_plan(plan,worker):
    devices=list(plan.get('devices',[]));count=int(plan.get('gpu_workers',len(devices)))
    if not devices or len(devices)!=len(set(devices)) or count!=len(devices):raise ValueError('Immutable unique device/worker membership required')
    if not isinstance(worker,str) or not worker.startswith('worker') or not worker[6:].isdigit():raise ValueError('Explicit workerN identifier required')
    index=int(worker[6:])
    if index>=count:raise ValueError('Worker index exceeds registered available devices')
    child=copy.deepcopy(plan);seeds=plan['seeds'][index::count]
    if not seeds:raise ValueError('Registered worker has no paired seeds')
    child['stage_id']=worker;child['seeds']=seeds;child['arms']=[a for a in plan['arms']if int(a['seed'])in seeds]
    if any(str(a['config']['device'])!=str(devices[index]) for a in child['arms']):raise ValueError('Worker/arm device differs from registered paired device')
    child['worker_index']=index;child['worker_device']=str(devices[index])
    return child

def checkpoint_record(runner,directory):
    manifest=runner.validate_training_manifest(directory);path=Path(directory)/manifest['latest'];record=C.file_record(path)
    return manifest,record

def timer_binding(plan,arm):
    return {'plan_sha256':C.digest(plan),'arm_sha256':C.digest(arm),'source_hashes':plan['source_hashes'],'budget_seconds':float(plan['active_wall_seconds_per_arm'])}

def validate_timer(timer,binding,manifest=None,record=None):
    if timer['binding']!=binding:raise ValueError('Registered equal-wall timer changed')
    if timer['state']=='IN_FLIGHT':raise RuntimeError('UNKNOWN active-wall fairness: invocation interrupted before atomic timer commitment; no time reset allowed')
    if timer['state']not in('READY','COMPLETE','UNKNOWN'):raise ValueError('Invalid timer state')
    if not math.isfinite(timer['active_seconds'])or timer['active_seconds']<0:raise ValueError('Invalid accumulated active time')
    if timer.get('checkpoint'):
        if record!=timer['checkpoint']or manifest['successful_updates']!=timer['successful_updates']:raise ValueError('Timer/model publication differs; UNKNOWN fairness, never restart')
    elif manifest is not None:raise ValueError('Unaccounted checkpoint exists before timed training')
    return timer

def train_arm(root,artifact,plan,bundle,arm,deadline_unix,max_invocations=None):
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['selected'])
    if sources!=plan['source_hashes']:raise ValueError('Frozen19 sources changed')
    C.environment(arm['config']);folder,directory=runner._arm_paths(root,artifact,plan,arm)
    timer_path=directory/'active_wall_timer.json';binding=timer_binding(plan,arm);directory.mkdir(parents=True,exist_ok=True)
    timer=C.read(timer_path)if timer_path.exists()else {'schema':'notebook19-active-wall-journal/v1','binding':binding,'state':'READY','active_seconds':0.,'successful_updates':0,'checkpoint':None,'invocations':[],'fair_wall_budget_verified':False}
    manifest,record=checkpoint_record(runner,directory)if(directory/'manifest.json').exists()else(None,None)
    validate_timer(timer,binding,manifest,record)
    if not timer_path.exists():C.atomic_json(timer_path,timer)
    budget=float(binding['budget_seconds']);tolerance=max(float(plan['wall_tolerance_seconds']),budget*float(plan['wall_tolerance_fraction']))
    if timer['state']=='COMPLETE':return final_arm(runner,root,artifact,plan,arm,timer)
    if timer['state']=='UNKNOWN':return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False,'timer':C.file_record(timer_path)}
    protocol=C.read(folder/'protocol.json')
    while timer['active_seconds']<budget:
        C.check_deadline(deadline_unix,5.)
        remaining=budget-timer['active_seconds'];before=int(timer['successful_updates'])
        if before>=int(plan['updates']):
            timer.update(state='UNKNOWN',reason='Scheduler update cap reached before registered active-wall budget; time parity unverified');C.atomic_json(timer_path,timer)
            return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False}
        invocation_deadline=min(float(deadline_unix),time.time()+remaining)
        marker={'sequence':len(timer['invocations'])+1,'started_unix':time.time(),'before_step':before,'before_checkpoint':timer['checkpoint'],'remaining_seconds':remaining}
        timer.update(state='IN_FLIGHT',in_flight=marker);C.atomic_json(timer_path,timer)
        started=time.perf_counter()
        try:
            result=runner._run_stage_model(plan,arm['kind'],arm['config'],bundle,folder,directory,protocol,
                model=runner._initial_model(root,artifact,plan,arm),stop_after=min(int(plan['updates']),before+int(plan['round_updates']),
                    min([m for m in map(int,plan['milestones'])if m>before],default=int(plan['updates']))),
                deadline_unix=invocation_deadline)
            manifest,record=checkpoint_record(runner,directory)
            if manifest['successful_updates']!=result['resume']['final_step']or manifest['attempted_updates']!=manifest['successful_updates']:raise ValueError('Nonfinite or failed updates invalidate timed confirmation')
            performance=copy.deepcopy(result['performance']);del result;gc.collect()
            elapsed=time.perf_counter()-started
            marker.update(ended_unix=time.time(),active_seconds=elapsed,after_step=manifest['successful_updates'],after_checkpoint=record,
                performance=performance)
            timer['active_seconds']+=elapsed;timer['successful_updates']=manifest['successful_updates'];timer['checkpoint']=record;timer['invocations'].append(marker)
            timer.pop('in_flight',None);timer['state']='READY';C.atomic_json(timer_path,timer)
            
        except BaseException:
            # Keep IN_FLIGHT explicitly. A full model checkpoint may be valid, but active wall between publications cannot be recovered honestly.
            raise
        if timer['active_seconds']>=budget:break
        if max_invocations is not None and len(timer['invocations'])>=int(max_invocations):
            return {'status':'READY_FOR_EXACT_RESUME','arm_id':arm['id'],'successful_updates':timer['successful_updates'],'timer':C.file_record(timer_path),'fair_wall_budget_verified':False}
    fair=budget<=timer['active_seconds']<=budget+tolerance and timer['successful_updates']<int(plan['updates'])
    timer.update(state='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,overshoot_seconds=max(0.,timer['active_seconds']-budget),
        reason='Registered equal active-wall budget reached'if fair else'Active-wall overshoot/update cap exceeded registered tolerance')
    C.atomic_json(timer_path,timer);return final_arm(runner,root,artifact,plan,arm,timer)

def matched_checkpoint(runner,directory,plan,actual_updates,timer=None):
    matched=int(plan['matched_updates']);directory=Path(directory)
    if int(actual_updates)<matched:return {'status':'UNKNOWN','updates':matched,'reason':'Registered matched-update endpoint not reached within unchanged active-wall budget'}
    saved=runner.validate_stage_milestones(directory,plan);record=saved['records'].get(str(matched))
    if not record or int(record.get('successful_updates',-1))!=matched or int(record.get('attempted_updates',-1))!=matched:raise ValueError('Exact matched checkpoint missing or invalid; never substitute another update')
    checked=C.file_record(record['path'])
    if checked['sha256']!=record['sha256']:raise ValueError('Pinned matched checkpoint hash differs')
    payload=torch.load(checked['path'],map_location='cpu',weights_only=False)
    if payload['state']['successful_updates']!=matched or payload['state']['attempted_updates']!=matched:raise ValueError('Pinned full-state step differs')
    matched_seconds=sum(float(entry['active_seconds'])for entry in(timer or{}).get('invocations',[])if int(entry['after_step'])<=matched)if timer else None
    return {'status':'COMPLETE','updates':matched,'source_encoder':checked,'active_wall_seconds':matched_seconds,
        'timing_scope':'Cumulative active-wall journal committed at the exact matched-update boundary',
        'milestone_manifest':C.file_record(directory/'milestones/manifest.json'),
        'selection':'Exact registered update from the same active-wall training pass; immutable outside latest-two/best rotation'}

def final_arm(runner,root,artifact,plan,arm,timer):
    folder,directory=runner._arm_paths(root,artifact,plan,arm);manifest,record=checkpoint_record(runner,directory)
    validate_timer(timer,timer_binding(plan,arm),manifest,record)
    return {'status':'COMPLETE'if timer['fair_wall_budget_verified']else'UNKNOWN','arm_id':arm['id'],'seed':arm['seed'],'variant':arm['variant'],'model_id':arm['kind'],
        'actual_updates':manifest['successful_updates'],'attempted_updates':manifest['attempted_updates'],'image_exposures':manifest['image_exposures'],
        'latest_checkpoint':record,'source_encoder':record,'initial_teacher':C.read(artifact/'stage_registration.json')['arms'][arm['id']]['initial_teacher'],
        'matched_checkpoint':matched_checkpoint(runner,directory,plan,manifest['successful_updates'],timer),
        'performance':copy.deepcopy(timer['invocations'][-1].get('performance',{}))if timer['invocations']else{},
        'fair_wall_budget_verified':timer['fair_wall_budget_verified'],'active_wall_seconds':timer['active_seconds'],'timer':C.file_record(directory/'active_wall_timer.json'),
        'schedule_updates':plan['updates'],'selection':'Last wall-budget checkpoint and pinned exact matched-update checkpoint; no best-DEV selection'}

def train_worker(root,artifact,plan,bundle,deadline_unix):
    C.bootstrap(root)
    from fmm_jepa.food101_training import run_writer
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['selected']);C.environment(plan['arms'][0]['config'])
    if len(plan['arms'])*float(plan['active_wall_seconds_per_arm'])>900 and not os.environ.get('TMUX'):raise RuntimeError('Long19 worker must run in detached tmux through the frontdoor')
    if sources!=plan['source_hashes']:raise ValueError('19 source binding changed')
    runner.prepare_stage(root,artifact,plan,bundle);preflight=runner.verify_stage_resume(root,artifact,plan,bundle,deadline_unix)
    if preflight['status']!='PASS':raise ValueError('Actual19 full-state exact-resume preflight failed')
    path=artifact/'training_result.json';saved=C.read(path)if path.exists()else {'status':'RUNNING','plan_hash':C.digest(plan),'arms':{},'fair_wall_budget_verified':False,'gate_receipts':plan['gate_receipts']}
    if saved['plan_hash']!=C.digest(plan):raise ValueError('Timed worker plan changed')
    with run_writer(artifact/'timed_training_writer'):
        for arm in plan['arms']:
            saved['arms'][arm['id']]=train_arm(root,artifact,plan,bundle,arm,deadline_unix);C.atomic_json(path,saved)
            if saved['arms'][arm['id']]['status']!='COMPLETE':saved['status']='UNKNOWN';C.atomic_json(path,saved);return saved
    values=list(saved['arms'].values());tolerance=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    fair=all(v['fair_wall_budget_verified']for v in values)and max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)<=tolerance
    saved.update(status='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,preflight=C.file_record(artifact/'stage_preflight.json'))
    C.atomic_json(path,saved);return saved


def primary_readout_row(row):
    """Optional kNN timeout never erases a certified, verified primary linear probe.

    Numerical or classifier convergence failures remain UNKNOWN. A recovered
    row keeps the original status and absent kNN value; it never fabricates a
    secondary result. Probe/prediction files must remain hash verified.
    """
    row=copy.deepcopy(row);original=row.get('status');knn_status=str(row.get('knn',{}).get('status',''))
    try:finite_primary=math.isfinite(float(row.get('accuracy_percent')))
    except (TypeError,ValueError):finite_primary=False
    records_valid=True
    for name in('probe_receipt','prediction_record'):
        record=row.get(name)
        if not isinstance(record,dict)or not record.get('path')or not record.get('sha256'):
            records_valid=False;break
        try:valid=C.file_record(record['path'])['sha256']==record['sha256']
        except OSError:valid=False
        if not valid:records_valid=False;break
    primary_valid=bool(row.get('converged'))and bool(row.get('numerical_valid'))and finite_primary and records_valid
    if original=='COMPLETE'and primary_valid:
        row['primary_readout_status']='COMPLETE';return row
    secondary_only=(original=='UNKNOWN_KNN_CUTOFF'or(original=='UNKNOWN_INCOMPLETE_OR_NONCONVERGED'and knn_status.startswith('UNKNOWN')))
    if secondary_only and primary_valid:
        row.update(status='COMPLETE',primary_readout_status='COMPLETE',secondary_knn_status=original,knn_accuracy_percent=None,
            secondary_knn_detail_status=knn_status,primary_scope='Certified fixed linear probe complete; descriptive kNN cutoff retained')
    else:
        row['primary_readout_status']='UNKNOWN'
        if original=='COMPLETE':row['status']='UNKNOWN_PRIMARY_READOUT'

    return row


def evaluate_worker(root,artifact,plan,bundle,training,deadline_unix):
    """TRAIN-fitted converged initial, exact matched-update, final DEV linear+kNN endpoints."""
    root,artifact=Path(root),Path(artifact);C.environment(plan['arms'][0]['config']);C.install_input_pipeline(int(plan['input_workers']),root)
    module=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm19_timed_evaluation')
    module.bind_model_builder(builder(root,plan['selected']),plan['source_hashes']);groups={'rows':[],'initial_rows':[],'matched_rows':[]}
    valid=True
    for arm in plan['arms']:
        saved=training['arms'][arm['id']]
        if saved['status']!='COMPLETE':valid=False;continue
        endpoints=[('initial',saved['initial_teacher'],0,None)]
        matched=saved['matched_checkpoint']
        if matched['status']=='COMPLETE':endpoints.append(('matched',matched['source_encoder'],matched['updates'],{'equal_updates':matched['updates']}))
        else:groups['matched_rows'].append({'status':'UNKNOWN','primary_readout_status':'UNKNOWN','arm_id':arm['id'],'seed':arm['seed'],'variant':arm['variant'],
            'endpoint_label':'matched','updates':matched['updates'],'accuracy_percent':None,'knn_accuracy_percent':None,'reason':matched['reason']})
        endpoints.append(('final',saved['source_encoder'],saved['actual_updates'],{'active_wall_seconds':saved['active_wall_seconds']}))
        for endpoint,record,count,cost in endpoints:
            C.check_deadline(deadline_unix)
            row=module.evaluate_endpoint(root,artifact/'evaluation'/endpoint,arm,bundle,record,count,deadline_unix,probe_config=plan['probe_convex'],
                endpoint='initial'if endpoint=='initial'else'final',training_cost=cost,run_id=plan['run_id'],stage_id=plan['stage_id']+'-'+endpoint)
            row=primary_readout_row(row);row.update(endpoint_label=endpoint,variant=arm['variant'],seed=arm['seed'])
            key='initial_rows'if endpoint=='initial'else'matched_rows'if endpoint=='matched'else'rows';groups[key].append(row)
            C.atomic_json(artifact/'evaluation_result.json',{'status':'RUNNING',**groups,'official_test':'CLOSED'})
            if endpoint!='matched'and(row.get('primary_readout_status')!='COMPLETE'or not row.get('converged')or not row.get('numerical_valid',False)):valid=False
    matched_complete=len(groups['matched_rows'])==len(plan['arms'])and all(row.get('primary_readout_status')=='COMPLETE'for row in groups['matched_rows'])
    final_complete=valid and len(groups['rows'])==len(plan['arms'])and len(groups['initial_rows'])==len(plan['arms'])
    result={'status':'COMPLETE'if final_complete else'UNKNOWN',**groups,'matched_status':'COMPLETE'if matched_complete else'UNKNOWN',
        'official_test':'CLOSED','source_hashes':plan['source_hashes'],'plan_hash':C.digest(plan),
        'scope':'Exploratory DEV; no official subset, seal, or confirmation query; absent matched/kNN results preserved UNKNOWN'}
    C.atomic_json(artifact/'evaluation_result.json',result);return result


def preflight(root,artifact,plan,bundle,deadline):
    backend,sources=bind(root,plan['selected'])
    if sources!=plan['source_hashes']:raise ValueError('Actual19 sources differ before preflight')
    backend.prepare_stage(root,artifact,plan,bundle)
    return backend.verify_stage_resume(root,artifact,plan,bundle,deadline)


def actual_preflight(root,artifact,selected,image_size=256,workers=0,deadline=None,device='cuda:0'):
    """Discarded actual GPU/config4-vs2+2 smoke, bounded600s; no learning/accuracy claim.

    Call after all source exports are frozen. Each resolution receives a separate
    unique artifact/run ID and discarded seed. Image preparation reads only TRAIN
    and development verification bytes; no readout or official test is opened.
    Reinvocation honors the original deadline and validates saved full state.
    """
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.mkdir(parents=True,exist_ok=True)
    if not artifact.is_relative_to(root/'artifacts/notebook_19'):raise ValueError('Actual preflight artifact belongs underartifacts/notebook_19/<unique_run_id>')
    relative=artifact.relative_to(root/'artifacts/notebook_19');run_id=relative.parts[0];resolution=int(image_size)
    if resolution not in(256,512):raise ValueError('Actual targetresolution256or512 required')
    request_path=artifact/'actual_preflight_request.json';selected={**C.canonical(selected),'image_size':resolution}
    original=C.read(request_path) if request_path.exists() else None
    if deadline is None:deadline=float(original['deadline_unix']) if original else time.time()+600.
    request={'schema':'notebook19-actual-gpu-preflight/v1','run_id':run_id,'resolution':resolution,'input_workers':int(workers),'selected':selected,
        'device':str(device),'deadline_unix':float(deadline),'source_hashes':C.source_hashes(root),'discarded_seed':20262298 if resolution==256 else 20262299,
        'quality_selection':'FORBIDDEN; discarded exact-resume proof only','official_test':'CLOSED'}
    C.immutable(request_path,request);started=time.perf_counter()
    try:
        C.check_deadline(deadline,5.)
        proposal={'seeds':[request['discarded_seed']],'schedule_updates':4000,'active_wall_seconds_per_arm':900.,'hard_limit_seconds':600.,'image_size':resolution,'matched_updates':200}
        plan=make_plan(root,run_id,proposal,selected,(str(device),),'actual-preflight'+str(resolution),int(workers),deadline,[])
        plan['evidence_role']='ACTUAL_CONFIG_DISCARDED_GPU_EXACT_RESUME_ONLY'
        C.immutable(artifact/'actual_preflight_plan.json',plan)
        bundle=data_bundle(root,artifact/('data_'+str(resolution)),plan['arms'][0]['config'],verify_images=True)
        certificate=preflight(root,artifact/('stage_'+str(resolution)),plan,bundle,deadline)
        result={'status':certificate['status'],'passed':certificate['status']=='PASS' and all(v.get('bitwise_equal',False) for v in certificate['arms'].values()),
            'certificate':certificate,'arms':certificate['arms'],'source_hashes':request['source_hashes'],'plan':C.file_record(artifact/'actual_preflight_plan.json'),
            'elapsed_seconds':time.perf_counter()-started,'image_size':resolution,'input_workers':int(workers),'scientific_training_launched':False,
            'actual_config':'FP32 dim192/depth4/head3, physical16 effective128, accumulation8,4000schedule; stop4 only',
            'quality_selection':'FORBIDDEN','official_test_pixels_read':0}
    except (RuntimeError,ValueError,TimeoutError,FloatingPointError) as error:
        result={'status':'UNKNOWN','passed':False,'error':repr(error),'source_hashes':request['source_hashes'],'elapsed_seconds':time.perf_counter()-started,
            'image_size':resolution,'input_workers':int(workers),'scientific_training_launched':False,'quality_selection':'FORBIDDEN','official_test_pixels_read':0}
    finally:
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    C.atomic_json(artifact/'actual_preflight_result.json',result);return result
